## 파일과 예외처리

- 텍스트 파일 읽고 쓰기
- 이진 파일 읽고 쓰기
- CSV 파일 읽고 쓰기
- 정규식을 사용하는 방법
- 예외를 처리하는 방법

# 파일의 기초
- 프로그램에서 만든 데이터를 영구히 저장하려면 하드 디스크에 파일 형태로 저장해야 한다.
- 파일은 보조기억장치에 기록된 논리적인 정보 단위이다.
  - 파일 안의 정보는 바이트 단위로 순차적으로 기록되어 있다.
  - 첫 번째 바이트의 위치를 오프셋(offset) 0이라고 하고 1씩 증가한다.
  - 파일을 읽고 쓰려면 먼저 파일을 열어야 한다.
  - 파일 포인터는 처음 열 때 0을 가리키며 읽거나 쓴 바이트 수만큼 증가한다.
  - 파일의 끝을 EOF(End of File)라고 한다.

파일의 종류
- 텍스트 파일: 각 바이트가 문자로 구성된 파일 (메모장으로 열면 글자가 보임)
- 이진 파일: 이진수의 바이트 배열이 저장된 파일 (메모장으로 열면 글자가 깨져 보임)

___
폴더(디렉토리)는 파일이나 서브폴더를 가질 수 있다.

경로(Path)
- 절대경로: `C:\user\song\PP2-2026\chap10\input.txt` 처럼 드라이브부터 전부 적은 경로
- 현재경로: 현재 프로그램이 실행 중인 폴더의 절대경로
- 상대경로: `.\chap10\input.txt` 처럼 현재경로를 기준으로 한 경로
  - `.` : 현재 디렉토리
  - `..` : 부모 디렉토리 (`../chap10/input.txt`)

파이썬 문자열에서 `\`는 특수문자라서 윈도우 경로는 `"d:\\input.txt"`처럼 두 번 쓰거나 `/`를 사용한다.

# 샘플 파일 만들기
- 아래 실습에서 읽을 파일들을 `data` 폴더에 미리 만들어 둔다.
- 이 셀을 먼저 실행해야 다음 셀들이 동작한다.

In [1]:
import os

os.makedirs("data", exist_ok=True)

def make_file(filename, text):
    with open(filename, "w", encoding="utf-8") as f:
        f.write(text)

make_file("data/input.txt", "홍길동\n김철수\n")
make_file("data/sales.txt", "1000000\n1000000\n1000000\n500000\n1500000\n")
make_file("data/proverbs.txt",
          "All's well that ends well.\n"
          "Bad news travels fast.\n"
          "Well begun is half done.\n"
          "Birds of a feather flock together.\n")
make_file("data/words.txt", "python\nfile\nerror\n")
make_file("data/test.txt", "abcdefghijklmnopqrstuvwxyz")
make_file("data/python.txt",
          "The joy of coding Python should be in seeing short\n"
          "readable classes.\n"
          "Python is executable pseudocode.\n")
make_file("data/uscons.txt",
          "Amendment XIII\n"
          "1. Neither slavery nor involuntary servitude, except as a punishment for crime\n"
          "whereof the party shall have been duly convicted, shall exist within the United States.\n"
          "2. Congress shall have power to enforce this article by appropriate\n"
          "legislation.\n")

print(sorted(os.listdir("data")))

['input.txt', 'proverbs.txt', 'python.txt', 'sales.txt', 'test.txt', 'uscons.txt', 'words.txt']


# 텍스트 파일 열고 닫기
`파일객체 = open(파일이름, 모드)` → 작업 → `파일객체.close()`

| 모드 | 의미 |
|---|---|
| `"r"` | 읽기 모드. 파일이 없으면 오류 |
| `"w"` | 쓰기 모드. 파일이 없으면 새로 만들고, 있으면 **기존 내용이 지워진다** |
| `"a"` | 추가 모드. 파일의 끝에 이어서 쓴다 |
| `"r+"` | 읽기와 쓰기 모드 |
| `"rb"`, `"wb"` | `b`를 붙이면 이진 파일 |

한글이 들어 있는 파일은 `encoding="utf-8"`을 지정한다. (UTF-8은 각 문자를 1~4바이트로 가변적으로 인코딩한다.)

# 텍스트 파일에서 한 줄 읽기
- `readline()`: `'\n'`까지 한 줄을 읽어서 문자열(str)로 반환. 파일의 끝이면 빈 문자열 `""`을 반환
- 읽은 줄 끝에는 `'\n'`이 붙어 있어서 `print()`하면 빈 줄이 하나씩 더 생긴다 → `rstrip()`으로 제거

In [2]:
infile = open("data/input.txt", "r", encoding="utf-8")
print(type(infile))             # 파일 객체의 타입

line = infile.readline()
while line != "":
    print(line)                 # 줄 끝의 '\n' 때문에 빈 줄이 생긴다.
    line = infile.readline()
infile.close()

print("--- rstrip() 사용 ---")
infile = open("data/input.txt", "r", encoding="utf-8")
line = infile.readline().rstrip()
while line != "":
    print(line)
    line = infile.readline().rstrip()
infile.close()

<class '_io.TextIOWrapper'>
홍길동

김철수

--- rstrip() 사용 ---
홍길동
김철수


# 텍스트 파일에 쓰기, 파일 닫기
- `write()`는 줄바꿈을 자동으로 넣지 않으므로 `"\n"`을 직접 붙인다.
- 파일을 닫는 3가지 방법
  1. `f.close()`를 직접 호출
  2. `try` ... `finally`에서 `f.close()` (예외가 발생해도 반드시 닫힘)
  3. `with open(...) as f:` → 블록을 빠져나오면 자동으로 닫힘 (**가장 간단하고 안전**)

In [3]:
# 1. close()를 직접 호출
outfile = open("data/output.txt", "w", encoding="utf-8")
outfile.write("김영희\n")
outfile.close()

# 2. try ~ finally
f = open("data/output.txt", "a", encoding="utf-8")     # 추가 모드
try:
    f.write("최자영\n")
finally:
    f.close()                   # 예외가 발생하더라도 반드시 실행된다.

# 3. with 문
with open("data/output.txt", "a", encoding="utf-8") as f:
    f.write("박민수\n")
print(f.closed)                 # 블록을 빠져나오면 자동으로 닫힌다.

with open("data/output.txt", "r", encoding="utf-8") as f:
    print(f.read())

True
김영희
최자영
박민수



# Lab: 매출 파일 처리
- 입력 파일(`sales.txt`)에 상점의 일별 매출이 저장되어 있다.
- 이것을 읽어서 총 매출과 일별 평균 매출을 계산한 후에 다른 파일(`summary.txt`)에 출력
- 입력 파일이 없을 수 있으므로 `try` ~ `except`로 묶는다.

In [4]:
def process_sales(infilename, outfilename):
    try:
        infile = open(infilename, "r", encoding="utf-8")
    except FileNotFoundError:
        print(f"파일 {infilename}을 발견할 수 없습니다.")
        return

    total = 0
    count = 0
    line = infile.readline()
    while line != "":
        total += int(line)      # 문자열을 정수형으로 바꾸기
        count += 1
        line = infile.readline()
    infile.close()

    outfile = open(outfilename, "w", encoding="utf-8")
    outfile.write("총매출 = " + str(total) + "\n")
    outfile.write("평균 일매출 = " + str(total / count))
    outfile.close()

def test_process_sales():
    process_sales("data/sales.txt", "data/summary.txt")
    with open("data/summary.txt", "r", encoding="utf-8") as f:
        print(f.read())
    process_sales("data/nofile.txt", "data/summary.txt")

if __name__ == "__main__":
    test_process_sales()

총매출 = 5000000
평균 일매출 = 1000000.0
파일 data/nofile.txt을 발견할 수 없습니다.


# 다양한 텍스트 입출력 방법
- `for line in infile:` → 파일 객체를 for 문에 넣으면 한 줄씩 읽는다.
- `str.strip()`: 양쪽의 공백 문자(스페이스, 탭, 줄바꿈)를 지운다. `lstrip()`은 왼쪽, `rstrip()`은 오른쪽만
  - `strip('#')`처럼 지울 문자를 지정할 수도 있다.
- `str.split()`: 문자열을 단어들의 리스트로 분리한다. 구분 문자를 안 주면 공백 문자로 나눈다.

In [5]:
s = " Hello, World!\n"
print(repr(s.strip()))          # 좌우의 공백 문자 지우기

s = "########this is example#####"
print(s.strip('#'))             # 좌우의 # 문자 지우기
print(s.lstrip('#'))
print(s.rstrip('#'))

print("a,b,c".split(","))       # 구분 문자 지정

infile = open("data/proverbs.txt", "r", encoding="utf-8")
for line in infile:
    line = line.rstrip()        # 오른쪽 공백 문자를 없앤다.
    word_list = line.split()    # 단어들로 분리한다.
    print(word_list)
infile.close()

'Hello, World!'
this is example
this is example#####
########this is example
['a', 'b', 'c']
["All's", 'well', 'that', 'ends', 'well.']
['Bad', 'news', 'travels', 'fast.']
['Well', 'begun', 'is', 'half', 'done.']
['Birds', 'of', 'a', 'feather', 'flock', 'together.']


# 파일 전체를 한꺼번에 읽기, 문자 단위로 읽기
- `read()`: 파일 전체를 하나의 문자열로 읽는다.
- `readlines()`: 파일 전체를 줄 단위의 리스트로 읽는다.
- `read(1)`: 문자 1개를 읽는다. 파일의 끝이면 `""`을 반환
- 주의: 파일이 거대하면 `read()`, `readlines()`는 메모리가 부족해질 수 있다.

In [6]:
infile = open("data/input.txt", "r", encoding="utf-8")
s = infile.read()
print(repr(s))
infile.close()

infile = open("data/input.txt", "r", encoding="utf-8")
lines = infile.readlines()
print(lines)
infile.close()

infile = open("data/input.txt", "r", encoding="utf-8")
ch = infile.read(1)
while ch != "":
    print(repr(ch), end=" ")
    ch = infile.read(1)
infile.close()

'홍길동\n김철수\n'
['홍길동\n', '김철수\n']
'홍' '길' '동' '\n' '김' '철' '수' '\n' 

# 문자 출현 횟수 계산
- 알파벳 A~Z가 각각 몇 번 나오는지 크기 26인 리스트에 센다.
- `ord(ch) - ord("A")`: 'A'는 0, 'B'는 1, ... 'Z'는 25

In [7]:
counter = [0] * 26

infile = open("data/proverbs.txt", "r", encoding="utf-8")
ch = infile.read(1)
while ch != "":
    ch = ch.upper()             # 소문자 -> 대문자
    if ch >= "A" and ch <= "Z":
        i = ord(ch) - ord("A")
        counter[i] += 1
    ch = infile.read(1)
infile.close()

print(counter)

[8, 3, 1, 4, 12, 5, 2, 4, 2, 0, 1, 11, 0, 4, 4, 0, 0, 4, 7, 7, 1, 1, 4, 0, 0, 0]


# Lab: 각 문자 횟수 세기
- 파일 안의 각 문자들이 몇 번이나 나타나는지를 딕셔너리로 센다. (키: 문자, 값: 횟수)

In [8]:
def count_chars(filename):
    freqs = {}
    infile = open(filename, "r", encoding="utf-8")
    for line in infile:
        for char in line.strip():       # 양쪽 끝의 공백 문자를 제거한다.
            if char in freqs:
                freqs[char] += 1        # 딕셔너리의 횟수를 증가한다.
            else:
                freqs[char] = 1         # 처음 나온 문자이면 1로 초기화한다.
    infile.close()
    return freqs

if __name__ == "__main__":
    print(count_chars("data/proverbs.txt"))

{'A': 1, 'l': 11, "'": 1, 's': 7, ' ': 16, 'w': 3, 'e': 12, 't': 7, 'h': 4, 'a': 7, 'n': 4, 'd': 4, '.': 4, 'B': 2, 'r': 4, 'v': 1, 'f': 5, 'W': 1, 'b': 1, 'g': 2, 'u': 1, 'i': 2, 'o': 4, 'c': 1, 'k': 1}


# Lab: 행맨
- 파일(`words.txt`)에서 단어를 하나 골라서 사용자가 맞히는 게임
- 한 번에 하나의 글자만 입력하며, 맞으면 글자가 보이고 틀리면 기회가 하나 줄어든다.
- `readlines()`로 읽은 단어 끝에는 `'\n'`이 붙어 있다. 그대로 두면 `'\n'`은 절대 맞힐 수 없어서 승리할 수 없으므로 `strip()`으로 없앤다.

In [9]:
import random

guesses = ''
turns = 10

infile = open("data/words.txt", "r", encoding="utf-8")
lines = infile.readlines()
infile.close()
word = random.choice(lines).strip()

while turns > 0:
    failed = 0
    for char in word:
        if char in guesses:
            print(char, end="")
        else:
            print("_", end="")
            failed += 1
    print("")
    if failed == 0:
        print("사용자 승리")
        break

    guess = input("단어를 추측하시오: ")
    guesses += guess
    if guess not in word:
        turns -= 1
        print("틀렸음!")
        print(str(turns) + "기회가 남았음!")
        if turns == 0:
            print("사용자 패배 정답은 " + word)

_____
단어를 추측하시오: e
e____
단어를 추측하시오: o
e__o_
단어를 추측하시오: r
error
사용자 승리


# Lab: CSV 파일 처리
- CSV는 테이블 형식의 데이터를 쉼표로 구분해서 저장하는 텍스트 파일 형식이다. (엑셀에서 열 수 있다.)
- `csv` 모듈: `csv.writer(f)`로 쓰고 `csv.reader(f)`로 읽는다. 한 줄이 리스트 하나가 된다.
- `next(data)`: 첫 줄(헤더)을 건너뛴다.
- 읽은 값은 모두 문자열이므로 숫자로 쓰려면 `float()`로 바꿔야 한다.
- 쓸 때는 `newline=""`을 지정해야 윈도우에서 빈 줄이 생기지 않는다.

날씨 정보를 읽어서 가장 추웠던 날을 조사 (강의자료에 나온 8일치만 사용)

In [10]:
import csv

rows = [
    ['날짜', '지점', '평균기온(℃)', '최저기온(℃)', '최고기온(℃)'],
    ['1980-04-01', 108, 6.5, 3.2, 11.7],
    ['1980-04-02', 108, 6.5, 1.4, 12.9],
    ['1980-04-03', 108, 11.1, 4.1, 18.4],
    ['1980-04-04', 108, 15.5, 8.6, 21],
    ['1980-04-05', 108, 15.4, 12.5, 18.2],
    ['1980-04-06', 108, 7.1, 4.3, 12.5],
    ['1980-04-07', 108, 8.5, 4.7, 13.3],
    ['1980-04-08', 108, 10.8, 8.4, 15.2],
]

# CSV 파일 쓰기
with open("data/weather.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    for row in rows:
        writer.writerow(row)

# CSV 파일 읽기
f = open("data/weather.csv", "r", encoding="utf-8", newline="")
data = csv.reader(f)
header = next(data)
print(header)

temp = 1000
day = ""
for row in data:
    print(row)
    if temp > float(row[3]):
        temp = float(row[3])
        day = row[0]
f.close()

print('가장 추웠던 날은', day, '최저기온은', temp, '입니다.')

['날짜', '지점', '평균기온(℃)', '최저기온(℃)', '최고기온(℃)']
['1980-04-01', '108', '6.5', '3.2', '11.7']
['1980-04-02', '108', '6.5', '1.4', '12.9']
['1980-04-03', '108', '11.1', '4.1', '18.4']
['1980-04-04', '108', '15.5', '8.6', '21']
['1980-04-05', '108', '15.4', '12.5', '18.2']
['1980-04-06', '108', '7.1', '4.3', '12.5']
['1980-04-07', '108', '8.5', '4.7', '13.3']
['1980-04-08', '108', '10.8', '8.4', '15.2']
가장 추웠던 날은 1980-04-02 최저기온은 1.4 입니다.


# Lab: 파일 암호화 (시저 암호)
- 알파벳을 n칸씩 밀어서 암호화하고, 반대로 밀어서 복호화한다.
- `key.index(l)`: 알파벳이 아닌 문자(공백, 마침표)는 `ValueError`가 발생 → `except`에서 그대로 붙인다.

In [11]:
key = 'abcdefghijklmnopqrstuvwxyz'

# 평문을 받아서 암호화하고 암호문을 반환한다.
def encrypt(n, plaintext):
    result = ''
    for l in plaintext.lower():
        try:
            i = (key.index(l) + n) % 26
            result += key[i]
        except ValueError:
            result += l
    return result

# 암호문을 받아서 복호화하고 평문을 반환한다.
def decrypt(n, ciphertext):
    result = ''
    for l in ciphertext:
        try:
            i = (key.index(l) - n) % 26
            result += key[i]
        except ValueError:
            result += l
    return result

def test_caesar():
    n = 3
    text = 'The language of truth is simple.'
    encrypted = encrypt(n, text)
    decrypted = decrypt(n, encrypted)
    print('평문: ', text)
    print('암호문: ', encrypted)
    print('복호문: ', decrypted)

if __name__ == "__main__":
    test_caesar()

평문:  The language of truth is simple.
암호문:  wkh odqjxdjh ri wuxwk lv vlpsoh.
복호문:  the language of truth is simple.


# 이진 파일
- 모드에 `b`를 추가해서 연다: `"rb"`, `"wb"`
- 이진 파일은 문자열이 아니라 `bytes`(바이트 배열)로 읽고 쓴다. 인코딩을 지정하지 않는다.
- 같은 123456이라도 텍스트로 저장하면 문자 '1'~'6'의 6바이트, 이진수로 저장하면 4바이트 정수

In [12]:
outfile = open("data/sample.bin", "wb")         # b를 추가
bytesArray = bytes([255, 128, 0, 1])            # 리스트 -> 바이트 배열
outfile.write(bytesArray)
outfile.close()

infile = open("data/sample.bin", "rb")
bytesArray = infile.read(8)                     # 최대 8바이트를 읽는다.
infile.close()
print(bytesArray)
print(bytesArray[0], list(bytesArray))          # 첫 번째 바이트, 전체

# 텍스트로 저장한 123456 과 이진수로 저장한 123456
with open("data/number.txt", "w") as f:
    f.write("123456")
with open("data/number.bin", "wb") as f:
    f.write((123456).to_bytes(4, "little"))

print(os.path.getsize("data/number.txt"), "바이트 /", os.path.getsize("data/number.bin"), "바이트")
with open("data/number.bin", "rb") as f:
    print(int.from_bytes(f.read(4), "little"))

b'\xff\x80\x00\x01'
255 [255, 128, 0, 1]
6 바이트 / 4 바이트
123456


# 순차접근과 임의접근
- 순차접근: 파일의 처음부터 순서대로 읽고 쓰는 것
- 임의접근: 파일 포인터를 원하는 위치로 옮겨서 읽고 쓰는 것
- `file.tell()`: 현재 파일 포인터의 위치를 반환
- `file.seek(오프셋, 기준위치)`: 기준위치로부터 오프셋만큼 이동

| 기준위치 | 의미 | 예 (현재 위치 100, 파일 크기 1000) |
|---|---|---|
| 0 | 파일 시작 기준 | `seek(10, 0)` → 10 |
| 1 | 현재 위치 기준 | `seek(10, 1)` → 110 |
| 2 | 파일 끝 기준 | `seek(-10, 2)` → 990 |

텍스트 모드에서는 시작 기준(0)으로만 자유롭게 이동할 수 있고, 현재/끝 기준으로 이동하려면 이진 모드로 열어야 한다.

In [13]:
infile = open("data/test.txt", "r+")
s = infile.read(10)                 # 파일 포인터 -> 10
print("읽은 문자열 : ", s)
position = infile.tell()
print("현재 위치: ", position)

infile.seek(0, 0)                   # 파일 포인터 -> 0
s = infile.read(10)
print("읽은 문자열 : ", s)
infile.close()

infile = open("data/test.txt", "rb")
infile.seek(10, 0)                  # 시작에서 10
print(infile.read(3), infile.tell())
infile.seek(5, 1)                   # 현재 위치(13)에서 5
print(infile.read(3), infile.tell())
infile.seek(-3, 2)                  # 끝에서 -3
print(infile.read(3), infile.tell())
infile.close()

읽은 문자열 :  abcdefghij
현재 위치:  10
읽은 문자열 :  abcdefghij
b'klm' 13
b'stu' 21
b'xyz' 26


# Lab: 파일 복사하기
- 이미지 같은 이진 파일을 다른 파일로 복사하는 `copyfile(원본, 복사본)` 함수
- 입력 파일에서 1024바이트씩 읽어서 출력 파일에 쓴다. 더 읽을 것이 없으면 빈 `bytes`가 반환되어 반복이 끝난다.
- 여기서는 이미지 대신 위에서 만든 이진 파일을 복사한다. (`123.png` 같은 이미지 파일도 같은 방법)

In [14]:
def copyfile(filename1, filename2):
    infile = open(filename1, "rb")
    outfile = open(filename2, "wb")

    # 입력 파일에서 1024 바이트씩 읽어서 출력 파일에 쓴다.
    while True:
        copy_buffer = infile.read(1024)
        if not copy_buffer:
            break
        outfile.write(copy_buffer)

    infile.close()
    outfile.close()
    print(filename1 + "를 " + filename2 + "로 복사하였습니다.")

def test_copyfile():
    with open("data/big.bin", "wb") as f:
        f.write(bytes(range(256)) * 10)         # 2560바이트짜리 이진 파일

    copyfile("data/big.bin", "data/copy.bin")
    print(os.path.getsize("data/big.bin"), os.path.getsize("data/copy.bin"))
    with open("data/big.bin", "rb") as f1, open("data/copy.bin", "rb") as f2:
        print(f1.read() == f2.read())

if __name__ == "__main__":
    test_copyfile()

data/big.bin를 data/copy.bin로 복사하였습니다.
2560 2560
True


# 객체 파일 저장 및 복원 (pickle)
- `pickle` 모듈의 `dump()`와 `load()`를 사용하면 객체를 파일에 쓰고 다시 읽을 수 있다.
- 리스트, 튜플, 딕셔너리, 세트 등 파이썬의 기본 데이터 타입을 그대로 저장할 수 있다.
- 이진수로 저장되므로 `"wb"`, `"rb"` 모드로 연다.

In [15]:
import pickle

gameOption = {
    "Sound": 8,
    "VideoQuality": "HIGH",
    "Money": 100000,
    "WeaponList": ["gun", "missile", "knife"]
}

file = open("data/save.p", "wb")        # 이진 파일 오픈
pickle.dump(gameOption, file)           # 딕셔너리를 피클 파일에 저장
file.close()

file = open("data/save.p", "rb")
obj = pickle.load(file)                 # 피클 파일에서 딕셔너리를 로딩
file.close()

print(obj)
print(type(obj), obj == gameOption)

{'Sound': 8, 'VideoQuality': 'HIGH', 'Money': 100000, 'WeaponList': ['gun', 'missile', 'knife']}
<class 'dict'> True


# 디렉토리 작업
`os` 모듈

| 함수 | 의미 |
|---|---|
| `os.getcwd()` | 현재 작업 디렉토리(current working directory)를 반환 |
| `os.chdir(경로)` | 작업 디렉토리를 변경 |
| `os.listdir()` | 디렉토리 안에 있는 파일과 폴더 이름의 리스트 |
| `os.path.isfile(이름)` | 파일인지 확인 |

- `str.startswith(접두어)`, `str.endswith(접미어)`로 파일 이름을 고를 수 있다.

In [16]:
import os

print(os.path.basename(os.getcwd()))    # 현재 작업 디렉토리의 폴더 이름

os.chdir("data")                        # data 폴더로 이동
print(os.path.basename(os.getcwd()))

# 작업 디렉토리에서 확장자가 ".txt"인 파일을 전부 찾아서 이름을 출력
for name in sorted(os.listdir()):
    if os.path.isfile(name):
        if name.endswith(".txt"):
            print(name)

os.chdir("..")                          # 원래 폴더로 돌아온다.

weak6
data
input.txt
number.txt
output.txt
proverbs.txt
python.txt
sales.txt
summary.txt
test.txt
uscons.txt
words.txt


# Lab: 디렉토리 안의 파일 처리
- 폴더 안의 텍스트 파일 중에서 "Python"을 포함하고 있는 줄이 있으면 파일의 이름과 해당 줄을 출력
- 강의자료의 `if "Python" in e:`는 변수 이름이 틀렸다 → 읽은 줄 `s`를 검사해야 한다.

In [17]:
def find_in_files(folder, keyword):
    for name in sorted(os.listdir(folder)):
        if not name.endswith(".txt"):
            continue
        infile = open(os.path.join(folder, name), "r", encoding="utf-8")   # 한글 처리
        for line in infile:
            s = line.rstrip()           # 오른쪽 줄바꿈 문자를 없앤다.
            if keyword in s:
                print(name, ":", s)
        infile.close()

if __name__ == "__main__":
    find_in_files("data", "Python")

python.txt : The joy of coding Python should be in seeing short
python.txt : Python is executable pseudocode.


# 정규식
- 정규식(regular expression)이란 특정한 규칙을 가지고 있는 문자열들을 메타 문자를 이용하여 표현하는 수식
- `re` 모듈 사용
  - `re.search(정규식, 문자열)`: 문자열 안에 규칙에 맞는 부분이 있으면 Match 객체, 없으면 `None`
  - `re.findall(정규식, 문자열)`: 규칙에 맞는 부분을 모두 찾아서 리스트로 반환

| 메타 문자 | 의미 |
|---|---|
| `.` | 아무 문자 1개 |
| `*` | 앞의 것이 0번 이상 반복 |
| `+` | 앞의 것이 1번 이상 반복 |
| `^` | 줄의 시작 |
| `$` | 줄의 끝 |
| `[0-9]`, `[a-z]` | 대괄호 안의 문자 중 하나 |
| `\d` | 숫자 (`[0-9]`와 같음) |

`\d`처럼 `\`가 들어가는 정규식은 `r"\d+"`처럼 raw 문자열로 쓴다. (그냥 `"\d+"`로 쓰면 경고가 나온다.)

In [18]:
import re

# 점과 별표: "X-Men"으로 시작하고 뒤에 아무 문자나 오는 제목
titles = ["X-Men: First Class", "X-Men: Days of Future Past", "X-Men Origins: Wolverine", "Iron Man"]
for title in titles:
    if re.search("^X-Men.*", title):
        print(title)

print("--- 숫자로 시작되는 줄 ---")
f = open("data/uscons.txt", "r", encoding="utf-8")
for line in f:
    line = line.rstrip()
    if re.search("^[0-9]+", line):
        print(line)
f.close()

print("--- 수강 번호만 추출 ---")
text = """101 COM PythonProgramming
102 MAT LinearAlgebra
103 ENG ComputerEnglish"""
print(re.findall(r"\d+", text))

X-Men: First Class
X-Men: Days of Future Past
X-Men Origins: Wolverine
--- 숫자로 시작되는 줄 ---
1. Neither slavery nor involuntary servitude, except as a punishment for crime
2. Congress shall have power to enforce this article by appropriate
--- 수강 번호만 추출 ---
['101', '102', '103']


# Lab: 패스워드 검사 프로그램
- 최소 8글자
- 적어도 하나의 소문자, 하나의 대문자
- 적어도 하나의 숫자
- 적어도 하나의 특수문자 `[_, @, $]`

In [19]:
import re

def check_password(password):
    if len(password) < 8:
        return False
    if not re.search("[a-z]", password):
        return False
    if not re.search("[A-Z]", password):
        return False
    if not re.search("[0-9]", password):
        return False
    if not re.search("[_@$]", password):
        return False
    return True

def test_check_password():
    for password in ["Python_2026", "Py_1", "python_2026", "PYTHON_2026", "Python_abcd", "Python2026"]:
        if check_password(password):
            print(password, ": 유효한 패스워드")
        else:
            print(password, ": 유효한 패스워드가 아닙니다.")

if __name__ == "__main__":
    test_check_password()

Python_2026 : 유효한 패스워드
Py_1 : 유효한 패스워드가 아닙니다.
python_2026 : 유효한 패스워드가 아닙니다.
PYTHON_2026 : 유효한 패스워드가 아닙니다.
Python_abcd : 유효한 패스워드가 아닙니다.
Python2026 : 유효한 패스워드가 아닙니다.


# 예외처리
- 사용자가 잘못된 데이터를 입력할 수도 있고, 열려는 파일이 없을 수도 있고, 0으로 나눌 수도 있다.
- 오류가 발생했을 때 프로그램이 그냥 죽지 않고, 오류를 알려주고 우아하게(gracefully) 종료하거나 계속 실행하도록 하는 것이 예외처리

오류의 종류
- 사용자 입력 오류: 정수를 입력해야 하는데 실수를 입력
- 장치 오류: 네트워크가 안 되거나 하드 디스크 작동 실패
- 코드 오류: 잘못된 인덱스로 리스트에 접근

| 예외 | 발생하는 때 |
|---|---|
| `ZeroDivisionError` | 0으로 나눌 때 |
| `ValueError` | 연산이나 함수의 인수가 적절하지 않은 값일 때 (`int("abc")`) |
| `IOError` (`OSError`) | 파일을 열 수 없을 때. 파일이 없으면 그 자식인 `FileNotFoundError` |
| `ImportError` | 모듈을 찾을 수 없을 때 |
| `KeyboardInterrupt` | 사용자가 Ctrl+C를 누를 때 |
| `EOFError` | 입력이 파일의 끝을 만났을 때 |

# try ~ except 구조
```
try:
    예외가 발생할 수 있는 문장
except 예외종류:
    예외를 처리하는 문장
```
- `try` 블록에서 예외가 발생하면 나머지 문장은 건너뛰고 바로 `except` 블록으로 간다.
- 예외가 발생하지 않으면 `except` 블록은 실행되지 않는다.
- `except 예외종류 as e:`로 받으면 오류 메시지를 `e`로 볼 수 있다.

In [20]:
(x, y) = (2, 0)
try:
    z = x / y
    print("이 줄은 실행되지 않는다.")
except ZeroDivisionError:
    print("0으로 나누는 예외")

try:
    z = x / y
except ZeroDivisionError as e:
    print(e)

0으로 나누는 예외
division by zero


In [21]:
# 정수가 입력될 때까지 다시 입력받기
while True:
    try:
        n = input("숫자를 입력하시오 : ")
        n = int(n)
        break
    except ValueError:
        print("정수가 아닙니다. 다시 입력하시오. ")

print("정수 입력이 성공하였습니다!")

숫자를 입력하시오 : 23.5
정수가 아닙니다. 다시 입력하시오. 
숫자를 입력하시오 : 10
정수 입력이 성공하였습니다!


In [22]:
try:
    fname = input("파일 이름을 입력하세요: ")
    infile = open(fname, "r")
except IOError:
    print("파일 " + fname + "을 발견할 수 없습니다.")

파일 이름을 입력하세요: kkk.py
파일 kkk.py을 발견할 수 없습니다.


# else 블록과 finally 블록
- `except`를 여러 개 써서 예외 종류마다 다르게 처리할 수 있다.
- `else`: 예외가 발생하지 **않았을 때만** 실행
- `finally`: 예외가 발생하든 안 하든 **반드시** 실행 (파일 닫기 같은 마무리 작업)

In [23]:
def divide(x, y):
    try:
        z = x / y
    except ZeroDivisionError:
        print("except: 0으로 나눌 수 없습니다.")
    except TypeError:
        print("except: 숫자가 아닙니다.")
    else:
        print("else: 결과는", z)
    finally:
        print("finally: 항상 실행")

def test_divide():
    divide(4, 2)
    divide(4, 0)
    divide(4, "a")

if __name__ == "__main__":
    test_divide()

else: 결과는 2.0
finally: 항상 실행
except: 0으로 나눌 수 없습니다.
finally: 항상 실행
except: 숫자가 아닙니다.
finally: 항상 실행


In [24]:
try:
    fh = open("data/testfile.txt", "w", encoding="utf-8")
    fh.write("테스트 데이터를 파일에 씁니다!!")
except IOError:
    print("Error: 파일을 찾을 수 없거나 데이터를 쓸 수 없습니다. ")
else:
    print("파일에 성공적으로 기록하였습니다. ")
    fh.close()

파일에 성공적으로 기록하였습니다. 


# 예외 발생하기
- `raise 예외종류(메시지)`로 강제로 예외를 발생시킬 수 있다.
- 함수 안에서 잘못된 값을 발견했을 때 호출한 쪽에 알려주는 용도로 쓴다.

In [25]:
def set_age(age):
    if age < 0:
        raise ValueError("나이는 음수가 될 수 없습니다.")
    return age

def test_raise():
    try:
        print(set_age(20))
        print(set_age(-1))
    except ValueError as e:
        print("오류:", e)

if __name__ == "__main__":
    test_raise()

20
오류: 나이는 음수가 될 수 없습니다.


# 이번 장에서 배운 것
- 파일을 읽을 때는 파일을 열고, 데이터를 읽은 후에, 파일을 닫는 절차가 필요하다. `with` 문을 쓰면 자동으로 닫힌다.
- 파일 모드에는 "r"(읽기), "w"(쓰기), "a"(추가)가 있다.
- 파일은 텍스트 파일과 이진 파일로 나누어진다. 이진 파일은 모드에 "b"를 붙인다.
- 파일에서 데이터를 읽거나 쓰는 함수는 `read()`와 `write()`이다. 텍스트 파일에서 한 줄씩 읽으려면 for 루프를 사용한다.
- `seek()`와 `tell()`로 파일 포인터를 옮겨서 임의접근을 할 수 있다.
- CSV는 `csv` 모듈, 객체 저장은 `pickle` 모듈, 문자열 규칙 검사는 `re` 모듈을 사용한다.
- 예외 처리는 오류가 발생했을 때 프로그램을 우아하게 종료하는 방법이다. `try` 블록과 `except` 블록으로 이루어지고, `else`, `finally`, `raise`를 함께 쓸 수 있다.